In [6]:
import plotly.graph_objects as go
import numpy as np
fig = go.Figure()
d_model=512
x=np.arange(d_model)
y=1/np.pow(10000,x/d_model)
fig.add_trace(go.Scatter(x=x,y=y,mode='lines'))
fig.show()


In [1]:
import sys
import random
import numpy as np
import os
from PIL import Image
from core.MyEnv import MyEnv
from lerobot.datasets.lerobot_dataset import LeRobotDataset

/Users/ningyu/code_before_paper/MyI10Tele/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Layout randomness: keep SEED=None and call `reset()` with no args between episodes
# so NumPy's RNG advances and cube pose / target position change each time.
# Pass an int to MyEnv(..., seed=K) only when you need a reproducible *first* scene;
# do not pass seed into every `reset()` during collection, or you repeat the same layout.


REPO_NAME = 'ningyv/auboI10'
NUM_DEMO = 10 # Number of demonstrations to collect
ROOT = "/Users/ningyu/code_before_paper/MyI10Tele/data2" # The root directory to save the demonstrations

In [3]:
I10_path = '/Users/ningyu/code_before_paper/MyI10Tele/assets/aubo_i10_2/aubo_i10.xml'
import mujoco 
model = mujoco.MjModel.from_xml_path(I10_path)
print(model.body_pos)

[[ 0.      0.      0.    ]
 [ 0.      0.      0.    ]
 [ 0.      0.      0.1632]
 [ 0.      0.2013  0.    ]
 [ 0.647   0.      0.    ]
 [ 0.6005  0.      0.    ]
 [ 0.      0.1025  0.    ]
 [ 0.     -0.094   0.    ]]


In [4]:
TASK_NAME = 'Put cube on the black platform' 
xml_path = '/Users/ningyu/code_before_paper/MyI10Tele/assets/aubo_i10_inspire/myscene.xml'
# xml_path = './asset/example_scene_y_i10.xml'
# Define the environment
PnPEnv = MyEnv(xml_path, seed=42)
print(f"action_type: {PnPEnv.action_type}")
print(f"state_type: {PnPEnv.state_type}")


-----------------------------------------------------------------------------
name:[myenv] dt:[0.001] HZ:[2000]
 n_qpos:[17] n_qvel:[16] n_qacc:[16] n_ctrl:[7]
 integrator:[IMPLICITFAST]

n_body:[19]
 [0/19] [world] mass:[0.00]kg
 [1/19] [base_link] mass:[2.59]kg
 [2/19] [shoulder_Link] mass:[10.18]kg
 [3/19] [upperArm_Link] mass:[18.10]kg
 [4/19] [foreArm_Link] mass:[4.45]kg
 [5/19] [wrist1_Link] mass:[1.79]kg
 [6/19] [wrist2_Link] mass:[1.63]kg
 [7/19] [wrist3_Link] mass:[0.20]kg
 [8/19] [i10_inspire_flange_link] mass:[0.05]kg
 [9/19] [gripper_base_link] mass:[0.08]kg
 [10/19] [wrist_cam_bracket] mass:[0.00]kg
 [11/19] [gripper_Link1] mass:[0.00]kg
 [12/19] [gripper_Link2] mass:[0.00]kg
 [13/19] [gripper_Link3] mass:[0.00]kg
 [14/19] [tcp_link] mass:[0.00]kg
 [15/19] [front_object_table] mass:[1.00]kg
 [16/19] [agentview_bracket] mass:[0.01]kg
 [17/19] [place_target_platform] mass:[0.08]kg
 [18/19] [cube] mass:[0.07]kg
body_total_mass:[40.23]kg

n_geom:[43]
geom_names:['floor', None

In [5]:
create_new = True
if os.path.exists(ROOT):
    print(f"Directory {ROOT} already exists.")
    ans = input("Do you want to delete it? (y/n) ")
    if ans == 'y':
        import shutil
        shutil.rmtree(ROOT)
    else:
        create_new = False


if create_new:
    dataset = LeRobotDataset.create(
                repo_id=REPO_NAME,
                root = ROOT, 
                robot_type="aubo_i10_inspire",
                fps=20, # 20 frames per second
                features={
                    "observation.image": {
                        "dtype": "video",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channels"],
                    },
                    "observation.wrist_image": {
                        "dtype": "video",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channel"],
                    },
                    "observation.state": {
                        "dtype": "float32",
                        # "shape": (7 if PnPEnv.state_type == 'qpos' else 6,),
                        "shape": (7 ,),
                        "names": ["state"], # 6 joint angles and 1 gripper ////  x, y, z, roll, pitch, yaw
                    },
                    "action": {
                        "dtype": "float32",
                        # "shape": (6 if PnPEnv.action_type == 'ee_pose' else 7,),
                        "shape": (7,),
                        "names": ["action"], # x, y, z, roll, pitch, yaw /// 6 joint angles and 1 gripper
                    },
                    "obj_init": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["obj_init"], # just the initial position of the object. Not used in training.
                    },
                },
                image_writer_threads=10,
                image_writer_processes=5,
        )
else:
    print("Load from previous dataset")
    dataset = LeRobotDataset(REPO_NAME, root=ROOT)

Directory /Users/ningyu/code_before_paper/MyI10Tele/data2 already exists.
Load from previous dataset


In [ ]:
action = np.zeros(7)
episode_id = 0
record_flag = False # Start recording when the robot starts moving
try:
    while PnPEnv.env.is_viewer_alive() and episode_id < NUM_DEMO:
        PnPEnv.step_env()
        if PnPEnv.env.loop_every(HZ=20):
            # check if the episode is done
            done = PnPEnv.check_success()
            if done: 
                # Save the episode data and reset the environment
                dataset.save_episode()
                PnPEnv.reset()
                episode_id += 1
                record_flag = False
            # Teleoperate the robot and get delta end-effector pose with gripper
            action, reset  = PnPEnv.teleop_robot()
            if not record_flag and sum(action) != 0:
                record_flag = True
                print("Start recording")
            if reset:
                # Reset the environment and clear the episode buffer
                # This can be done by pressing 'z' key
                PnPEnv.reset()
                dataset.clear_episode_buffer()
                record_flag = False
            # Step the environment
            # Get the end-effector pose and images
            # obs_action = PnPEnv.get_ee_pose()
            obs_action=PnPEnv.get_obs_action()
            # assert obs_action.type == PnPEnv.action_type , print(f"expect action_type: {PnPEnv.action_type}, but got {obs_action.type}")
            assert obs_action.type == "qpos"
            agent_image,wrist_image = PnPEnv.grab_image()
            # # resize to 256x256
            agent_image = Image.fromarray(agent_image)
            wrist_image = Image.fromarray(wrist_image)
            agent_image = agent_image.resize((256, 256))
            wrist_image = wrist_image.resize((256, 256))
            agent_image = np.array(agent_image)
            wrist_image = np.array(wrist_image)
            obs_state = PnPEnv.step(action)
    
            # from IPython.display import display, clear_output
            # clear_output(wait=True)
            # print(f"gripper_qpos: {PnPEnv.env.get_qpos_joint('rh_r1')}") # close : 0.81454458 open :2.7e-6
            # print(f"gripper_qpos: {PnPEnv.env.get_qpos_joint('rh_r1')[0]}")
            
            assert obs_state.type == PnPEnv.state_type, f"expect state_type: {PnPEnv.state_type}, but got {obs_state.type}"
            if record_flag:
                # Add the frame to the dataset
                dataset.add_frame({
                    "observation.image": agent_image,
                    "observation.wrist_image": wrist_image,
                    "observation.state": obs_state,
                    "action": obs_action,
                    "obj_init": PnPEnv.obj_init_pose,
                    "task": TASK_NAME,
                })
                # print(PnPEnv.obj_init_pose)
                print(f"cube pos {PnPEnv.env.get_p_body('cube')}")
                print(f"target pos {PnPEnv.env.get_p_body('place_target_platform')}")
            PnPEnv.render(teleop=True)

except Exception as e:
    print(f"Interrupted: {e}")
finally:
    PnPEnv.env.close_viewer()
    dataset.stop_image_writer()
    dataset.finalize()

Start recording
cube pos [ 0.36495785 -1.10611216  0.2559325 ]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598251]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598539]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1

2026-05-13 12:08:49.929 python[87054:1515300] error messaging the mach port for IMKCFRunLoopWakeUpReliable


cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.255

2026-05-13 12:08:50.866 python[87054:1515300] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit


cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.25598548]
target pos [ 0.16868712 -1.07388603  0.238     ]
cube pos [ 0.36495785 -1.10611216  0.255

...

Map: 100%|██████████| 736/736 [00:00<00:00, 3920.42 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	Apple LLVM 15.0.0 (clang-1500.3.9.4)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:06:26
Svt[info]: -------------------------------------------
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	Apple LLVM 15.0.0 (clang-1500.3.9.4)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:06:26
Svt[info]: -------------------------------------------
Svt[warn]: Preset M12 is mapped to M10.
Svt[info]: Level of Parallelism: 4
Svt[info]: Number of PPCS 59
Svt[info]: [asm level on system : up to neon_dotprod]
Svt[info]: [asm level selected : up to neon_dotprod]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / f

DONE INITIALIZATION
Start recording
cube pos [ 0.29784904 -1.0529302   0.25593238]
target pos [ 0.11779699 -1.14561962  0.238     ]
cube pos [ 0.29784904 -1.0529302   0.2559824 ]
target pos [ 0.11779699 -1.14561962  0.238     ]
cube pos [ 0.29784904 -1.0529302   0.25598539]
target pos [ 0.11779699 -1.14561962  0.238     ]
cube pos [ 0.29784904 -1.0529302   0.25598548]
target pos [ 0.11779699 -1.14561962  0.238     ]
cube pos [ 0.29784904 -1.0529302   0.25598548]
target pos [ 0.11779699 -1.14561962  0.238     ]
cube pos [ 0.29784904 -1.0529302   0.25598548]
target pos [ 0.11779699 -1.14561962  0.238     ]
cube pos [ 0.29784904 -1.0529302   0.25598548]
target pos [ 0.11779699 -1.14561962  0.238     ]
cube pos [ 0.29784904 -1.0529302   0.25598548]
target pos [ 0.11779699 -1.14561962  0.238     ]
cube pos [ 0.29784904 -1.0529302   0.25598548]
target pos [ 0.11779699 -1.14561962  0.238     ]
cube pos [ 0.29784904 -1.0529302   0.25598548]
target pos [ 0.11779699 -1.14561962  0.238     ]
cube

Map: 100%|██████████| 656/656 [00:00<00:00, 4311.02 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	Apple LLVM 15.0.0 (clang-1500.3.9.4)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:06:26
Svt[info]: -------------------------------------------
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	Apple LLVM 15.0.0 (clang-1500.3.9.4)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:06:26
Svt[info]: -------------------------------------------
Svt[warn]: Preset M12 is mapped to M10.
Svt[info]: Level of Parallelism: 4
Svt[info]: Number of PPCS 59
Svt[info]: [asm level on system : up to neon_dotprod]
Svt[info]: [asm level selected : up to neon_dotprod]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / f

DONE INITIALIZATION
Start recording
cube pos [ 0.29325694 -1.11295403  0.25593238]
target pos [ 0.08411704 -1.10242951  0.238     ]
cube pos [ 0.29325694 -1.11295403  0.2559824 ]
target pos [ 0.08411704 -1.10242951  0.238     ]
cube pos [ 0.29325694 -1.11295403  0.25598539]
target pos [ 0.08411704 -1.10242951  0.238     ]
cube pos [ 0.29325694 -1.11295403  0.25598548]
target pos [ 0.08411704 -1.10242951  0.238     ]
cube pos [ 0.29325694 -1.11295403  0.25598548]
target pos [ 0.08411704 -1.10242951  0.238     ]
cube pos [ 0.29325694 -1.11295403  0.25598548]
target pos [ 0.08411704 -1.10242951  0.238     ]
cube pos [ 0.29325694 -1.11295403  0.25598548]
target pos [ 0.08411704 -1.10242951  0.238     ]
cube pos [ 0.29325694 -1.11295403  0.25598548]
target pos [ 0.08411704 -1.10242951  0.238     ]
cube pos [ 0.29325694 -1.11295403  0.25598548]
target pos [ 0.08411704 -1.10242951  0.238     ]
cube pos [ 0.29325694 -1.11295403  0.25598548]
target pos [ 0.08411704 -1.10242951  0.238     ]
cube

: 